In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Dict, List

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 13 — FUNCTION 3 (RL-Inspired Exploration–Exploitation v2)
#
# What changed vs Week 12:
#  1) RL-style policy over "search modes" (arms):
#       - GLOBAL (Sobol), PCA-guided, LOCAL refinement
#     We estimate each mode's expected reward (EI/UCB proxy) and
#     allocate candidate budget adaptively (MAB/UCB + epsilon-greedy).
#
#  2) Offline Q-learning from the historical trajectory:
#       - State: recent improvement regime (improving vs stagnating)
#       - Actions: {EXPLORE, BALANCED, EXPLOIT}
#       - Reward: increment in best-so-far after each observation
#     Learned Q biases acquisition parameters:
#       - EXPLORE => higher beta / higher xi / more global+PCA
#       - EXPLOIT => lower beta / lower xi / more local
#
#  3) Faster convergence / efficiency:
#       - Two-stage selection: (a) probe each mode cheaply, (b) spend
#         most budget on the best modes, (c) final global argmax score.
#
#  4) Still enforces: bounds, min distance, repulsion, PC redundancy.
#  5) Output: a single next point rounded to <= 6 decimals.
# ============================================================

DEVICE = torch.device("cpu")  # set "cuda" if available

# ---------------------------------------------------------
# 1. Data: 3D inputs and 1D outputs (maximisation)
# ---------------------------------------------------------
X_raw = np.array([
    [0.17152521, 0.34391687, 0.2487372],
    [0.24211446, 0.64407427, 0.27243281],
    [0.53490572, 0.39850092, 0.17338873],
    [0.49258141, 0.61159319, 0.34017639],
    [0.13462167, 0.21991724, 0.45820622],
    [0.34552327, 0.94135983, 0.26936348],
    [0.15183663, 0.43999062, 0.99088187],
    [0.64550284, 0.39714294, 0.91977134],
    [0.74691195, 0.28419631, 0.22629985],
    [0.17047699, 0.6970324 , 0.14916943],
    [0.22054934, 0.29782524, 0.34355534],
    [0.66601366, 0.67198515, 0.2462953 ],
    [0.04680895, 0.23136024, 0.77061759],
    [0.60009728, 0.72513573, 0.06608864],
    [0.96599485, 0.86111969, 0.56682913],
    [1.065994  , 1.041359  , 1.090881  ],  # historical out-of-bounds (kept; clipped)
    [0.403482  , 0.38217   , 0.489363  ],
    [3.98350e-01, 1.00000e-06, 5.43642e-01],
    [0.962851  , 0.987386  , 0.040875  ],
    [0.504564  , 0.348726  , 0.601264  ],
    [0.265159  , 0.286931  , 0.413777  ],
    [0.403756  , 0.381706  , 0.489738  ],
    [0.359927  , 0.175969  , 0.720956  ],
    [0.846635  , 0.969038  , 0.008713  ],
    [0.952304, 0.517975, 0.686149],
    [0.349881, 0.437030, 0.503435],
    [0.990478, 0.920657, 0.490369]
], dtype=float)

y_raw = np.array([
    -0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758,
    -0.11062091, -0.39892551, -0.11386851, -0.13146061, -0.09418956,
    -0.04694741, -0.10596504, -0.11804826, -0.03637783, -0.05675837,
    -0.769427956661122, -0.03310307977430594, -0.09333459499358941,
    -0.07627377706316849, -0.05678719487656195, -0.03492633073917894,
    -0.009136026447950633, -0.14476549871155003, -0.11906253814017263,
    -0.1409808967165733, -0.03118483065316241, -0.031887886470039595
], dtype=float)

# ---------------------------------------------------------
# Utility: bounds + rounding for submission
# ---------------------------------------------------------
def clip_to_bounds(X: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    return np.clip(X, lower, upper)

def round6(x: np.ndarray) -> np.ndarray:
    return np.round(x.astype(float), 6)

def as_fixed6_list(x: np.ndarray) -> str:
    return f"[{x[0]:.6f}, {x[1]:.6f}, {x[2]:.6f}]"

def frac_out_of_bounds(X: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> float:
    bad = ((X < lower) | (X > upper)).any(axis=1)
    return float(np.mean(bad))

# ---------------------------------------------------------
# 2. PyTorch MLP model
# ---------------------------------------------------------
class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes=(64, 64), dropout=0.15):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def _train_mlp(
    Xs: np.ndarray,
    ys: np.ndarray,
    hidden: Tuple[int, ...],
    dropout: float,
    lr: float,
    weight_decay: float,
    n_epochs: int,
    seed: int,
    tol: float = 1e-6,
    patience: int = 80,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MLPRegressorTorch(
        input_dim=Xs.shape[1],
        hidden_sizes=hidden,
        dropout=dropout
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
    y_tensor = torch.from_numpy(ys.astype(np.float32)).view(-1, 1).to(DEVICE)

    best_loss = float("inf")
    bad = 0

    for _ in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_tensor)
        loss = criterion(preds, y_tensor)
        loss.backward()
        optimizer.step()

        l = float(loss.item())
        if best_loss - l > tol:
            best_loss = l
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    return model

# ---------------------------------------------------------
# 3. MC Dropout surrogate (robust y scaling)
# ---------------------------------------------------------
@dataclass
class MCDropoutSurrogate:
    hidden_layer_sizes: Tuple[int, ...] = (64, 64)
    dropout: float = 0.15
    n_epochs: int = 1200
    lr: float = 1e-3
    weight_decay: float = 1e-6
    random_state: int = 0
    n_mc_samples: int = 128
    robust_y: bool = True

    def __post_init__(self):
        self.model = None
        self.x_scaler = StandardScaler()
        self.y_scaler = RobustScaler() if self.robust_y else StandardScaler()

    def fit(self, X: np.ndarray, y: np.ndarray, lower: np.ndarray, upper: np.ndarray):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.fit_transform(Xc)
        ys = self.y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

        self.model = _train_mlp(
            Xs, ys,
            hidden=self.hidden_layer_sizes,
            dropout=self.dropout,
            lr=self.lr,
            weight_decay=self.weight_decay,
            n_epochs=self.n_epochs,
            seed=self.random_state,
        )

    def predict(self, X: np.ndarray, lower: np.ndarray, upper: np.ndarray, return_std: bool = False):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.transform(Xc)
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)

        # Keep dropout ON at inference for epistemic uncertainty
        self.model.train()

        preds_scaled_mc = []
        with torch.no_grad():
            for _ in range(self.n_mc_samples):
                preds_scaled_mc.append(self.model(X_tensor).cpu().numpy().ravel())

        preds_scaled_mc = np.stack(preds_scaled_mc, axis=0)
        mean_scaled = preds_scaled_mc.mean(axis=0)
        std_scaled = preds_scaled_mc.std(axis=0)

        scale_y = float(self.y_scaler.scale_[0])
        center_y = float(self.y_scaler.center_[0])

        mean = mean_scaled * scale_y + center_y
        if not return_std:
            return mean

        std = std_scaled * abs(scale_y)
        return mean, std

# ---------------------------------------------------------
# 3b. Ensemble wrapper
# ---------------------------------------------------------
@dataclass
class EnsembleSurrogate:
    members: List[MCDropoutSurrogate]

    def fit(self, X: np.ndarray, y: np.ndarray, lower: np.ndarray, upper: np.ndarray):
        for m in self.members:
            m.fit(X, y, lower, upper)

    def predict(self, X: np.ndarray, lower: np.ndarray, upper: np.ndarray, return_std: bool = False):
        mus, sigs = [], []
        for m in self.members:
            mu, sig = m.predict(X, lower, upper, return_std=True)
            mus.append(mu)
            sigs.append(sig)

        mus = np.stack(mus, axis=0)
        sigs = np.stack(sigs, axis=0)

        mu_ens = mus.mean(axis=0)
        var_within = (sigs ** 2).mean(axis=0)
        var_between = mus.var(axis=0)
        sig_ens = np.sqrt(np.maximum(var_within + var_between, 1e-12))

        if not return_std:
            return mu_ens
        return mu_ens, sig_ens

# ---------------------------------------------------------
# 4. Acquisition helpers
# ---------------------------------------------------------
def acquisition_pi_ei(mu: np.ndarray, sigma: np.ndarray, y_best: float, xi: float = 0.0):
    sigma = np.maximum(sigma, 1e-9)
    gamma = (mu - y_best - xi) / sigma
    pi = norm.cdf(gamma)
    ei = (mu - y_best - xi) * pi + sigma * norm.pdf(gamma)
    return pi, np.maximum(ei, 0.0)

# ---------------------------------------------------------
# 5. Week 13: RL policy layer (offline Q-learning + MAB allocation)
# ---------------------------------------------------------
def _best_so_far_series(y: np.ndarray) -> np.ndarray:
    b = np.empty_like(y, dtype=float)
    m = -np.inf
    for i in range(len(y)):
        m = max(m, float(y[i]))
        b[i] = m
    return b

def _build_states_from_history(y: np.ndarray, window: int = 5, eps: float = 1e-12) -> np.ndarray:
    """
    State 0: stagnating (no meaningful improvement recently)
    State 1: improving (recent improvement exists)
    """
    bsf = _best_so_far_series(y)
    imp = np.diff(bsf, prepend=bsf[0])
    states = np.zeros(len(y), dtype=int)
    for t in range(len(y)):
        lo = max(0, t - window + 1)
        recent = imp[lo:t+1]
        states[t] = int(np.max(recent) > eps)
    return states

def offline_q_learning_policy(
    y: np.ndarray,
    gamma: float = 0.90,
    lr: float = 0.25,
    window: int = 5,
) -> Dict:
    """
    Actions:
      0 = EXPLORE, 1 = BALANCED, 2 = EXPLOIT
    Reward: best-so-far improvement at each step.
    We do a simple heuristic for action attribution:
      - when state is stagnating -> prefer explore,
      - when improving -> prefer exploit,
    and let Q-learning refine the preference magnitudes.
    """
    y = np.asarray(y, float)
    n = len(y)
    states = _build_states_from_history(y, window=window)
    bsf = _best_so_far_series(y)
    r = np.diff(bsf, prepend=bsf[0])  # reward at each step

    Q = np.zeros((2, 3), dtype=float)

    for t in range(n - 1):
        s = int(states[t])
        # heuristic behavior policy (offline):
        # stagnating => explore, improving => exploit, else balanced occasionally
        if s == 0:
            a = 0
        else:
            a = 2
        # small chance of balanced for stability in updates (deterministic tie-break)
        if (t % 7) == 0:
            a = 1

        s2 = int(states[t + 1])
        target = float(r[t + 1]) + gamma * float(np.max(Q[s2]))
        Q[s, a] = (1.0 - lr) * Q[s, a] + lr * target

    # current state is last regime
    s_now = int(states[-1])
    a_star = int(np.argmax(Q[s_now]))

    # map action -> parameter multipliers
    # (values chosen to be stable, not extreme)
    if a_star == 0:       # EXPLORE
        params = dict(xi_mult=1.45, beta_mult=1.30, alpha_ucb=0.40, local_frac=0.18)
        action_name = "EXPLORE"
    elif a_star == 2:     # EXPLOIT
        params = dict(xi_mult=0.75, beta_mult=0.90, alpha_ucb=0.22, local_frac=0.42)
        action_name = "EXPLOIT"
    else:                 # BALANCED
        params = dict(xi_mult=1.00, beta_mult=1.00, alpha_ucb=0.30, local_frac=0.30)
        action_name = "BALANCED"

    return dict(Q=Q, state_now=s_now, action=a_star, action_name=action_name, params=params)

def _mab_allocate_budgets_ucb(
    mode_scores: Dict[str, float],
    total: int,
    t: int,
    explore_frac: float = 0.10,
) -> Dict[str, int]:
    """
    Multi-armed bandit style allocation (UCB-like) over 3 modes.
    mode_scores is a proxy for expected reward (bigger is better).
    We allocate most budget proportional to softmax(mode_scores),
    plus a small exploration floor.
    """
    modes = list(mode_scores.keys())
    s = np.array([mode_scores[m] for m in modes], dtype=float)

    # stabilize
    s = s - np.max(s)
    w = np.exp(s)
    w = w / max(float(np.sum(w)), 1e-12)

    floor = int(total * explore_frac / len(modes))
    remaining = total - floor * len(modes)
    alloc = {m: floor for m in modes}
    add = np.floor(remaining * w).astype(int)
    for i, m in enumerate(modes):
        alloc[m] += int(add[i])

    # fix rounding leftovers
    used = sum(alloc.values())
    for m in modes:
        if used >= total:
            break
        alloc[m] += 1
        used += 1

    return alloc

# ---------------------------------------------------------
# 6. Week 13 proposer (RL-tuned)
# ---------------------------------------------------------
def propose_next_point_week13(
    surrogate: EnsembleSurrogate,
    X_obs: np.ndarray,
    y_obs: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 123,

    # Total candidate budget (adaptive split across modes)
    n_total: int = 110_000,

    # Distance / redundancy controls
    min_dist: float = 0.010,
    repulse_len: float = 0.050,
    repulse_w: float = 0.28,
    bound_margin: float = 0.006,

    # Base acquisition controls (will be RL-adjusted)
    xi_base: float = 0.0008,
    alpha_ucb_base: float = 0.30,

    # PCA controls
    pca_dims: int = 2,
    pca_bonus_w: float = 0.020,
    pc_redundancy_w: float = 0.020,

    # Local refinement base
    local_sigma: float = 0.040,

    # Mode-probing (cheap evaluation before allocating full budgets)
    probe_per_mode: int = 6000,

    # epsilon-greedy on mode choice (small, final round still mostly exploit)
    eps_mode: float = 0.05,

    enforce_rounded_nonduplicate: bool = True,
) -> Dict:
    rng = np.random.RandomState(random_state)
    lower, upper = np.asarray(bounds[0], float), np.asarray(bounds[1], float)

    Xc = clip_to_bounds(X_obs, lower, upper)
    best_idx = int(np.argmax(y_obs))
    y_best = float(y_obs[best_idx])
    x_best = Xc[best_idx].copy()

    n = len(y_obs)
    d = Xc.shape[1]

    # -------- RL/Q-learning: choose EXPLORE vs EXPLOIT regime --------
    policy = offline_q_learning_policy(y_obs, gamma=0.90, lr=0.25, window=5)
    xi = float((xi_base + 0.006 / np.sqrt(max(n, 1))) * policy["params"]["xi_mult"])
    alpha_ucb = float(policy["params"]["alpha_ucb"])
    beta_base = float(0.55 * np.sqrt(np.log(n + 2.0)))
    beta = float(beta_base * policy["params"]["beta_mult"])

    # -------- PCA lens on standardized space --------
    x_scaler = StandardScaler()
    Xs = x_scaler.fit_transform(Xc)

    pca = PCA(n_components=d, random_state=random_state)
    Z = pca.fit_transform(Xs)
    evr = pca.explained_variance_ratio_
    comps = pca.components_
    k = int(np.clip(pca_dims, 1, d))

    top_idx = np.argsort(y_obs)[::-1][:min(4, n)]
    Z_top = Z[top_idx, :k]
    Z_mean = Z.mean(axis=0)

    step = np.sqrt(np.maximum(pca.explained_variance_[:k], 1e-12))
    step = step / np.maximum(np.mean(step), 1e-12)

    # -------- Candidate generators (three "arms") --------
    def gen_sobol(m: int) -> np.ndarray:
        sob = torch.quasirandom.SobolEngine(dimension=d, scramble=True, seed=int(random_state))
        X_u = sob.draw(m).cpu().numpy()
        return lower + (upper - lower) * X_u

    def gen_pca(m: int) -> np.ndarray:
        anchors = Z_top[rng.randint(len(Z_top), size=m)]
        noise = rng.normal(0.0, 1.0, size=(m, k)) * (0.85 * step.reshape(1, -1))
        Zcand_k = anchors + noise

        Zfull = np.tile(Z_mean.reshape(1, -1), (m, 1))
        Zfull[:, :k] = Zcand_k
        Xp = x_scaler.inverse_transform(pca.inverse_transform(Zfull))
        return Xp

    def gen_local(m: int, sigma: float) -> np.ndarray:
        return x_best + rng.normal(0.0, sigma, size=(m, d))

    # -------- Probe each mode to estimate "expected reward" --------
    # reward proxy = mean of top-q (EI_norm + alpha*UCB_impr) after constraints
    def score_batch(Xcand: np.ndarray) -> Tuple[np.ndarray, Dict]:
        Xcand = clip_to_bounds(Xcand, lower, upper)
        mu, sig = surrogate.predict(Xcand, lower, upper, return_std=True)
        pi, ei = acquisition_pi_ei(mu, sig, y_best=y_best, xi=xi)

        # distances / constraints
        dmat = np.linalg.norm(Xcand[:, None, :] - Xc[None, :, :], axis=2)
        dmin = dmat.min(axis=1)
        ok = dmin >= min_dist

        repulse_pen = np.exp(-(dmin ** 2) / (2.0 * repulse_len ** 2))

        dist_to_lower = (Xcand - lower)
        dist_to_upper = (upper - Xcand)
        bound_close = np.minimum(dist_to_lower, dist_to_upper).min(axis=1)
        bound_pen = np.exp(- (bound_close / max(bound_margin, 1e-9)) ** 2)

        ucb_impr = np.maximum(mu + beta * sig - y_best, 0.0)
        ei_max = float(np.max(ei[np.isfinite(ei)]) if np.isfinite(ei).any() else 1.0)
        ei_norm = ei / max(ei_max, 1e-12)

        comp_ei = (1.0 - alpha_ucb) * ei_norm
        comp_ucb = alpha_ucb * ucb_impr
        comp_repulse = repulse_w * repulse_pen
        comp_bound = 0.10 * bound_pen

        # PCA bonus + PC redundancy
        Zcand_all = pca.transform(x_scaler.transform(Xcand))[:, :k]
        Z_std = np.std(Z[:, :k], axis=0) + 1e-12
        z_mag = np.mean(np.abs(Zcand_all) / Z_std.reshape(1, -1), axis=1)
        pca_bonus = pca_bonus_w * (1.0 - np.exp(-z_mag))

        Z_obs_k = Z[:, :k]
        d_pc = np.linalg.norm(Zcand_all[:, None, :] - Z_obs_k[None, :, :], axis=2)
        d_pc_min = d_pc.min(axis=1)
        pc_redundancy_pen = pc_redundancy_w * np.exp(-(d_pc_min ** 2) / (2.0 * 0.45 ** 2))

        score = comp_ei + comp_ucb + pca_bonus - comp_repulse - comp_bound - pc_redundancy_pen
        score = np.where(ok, score, -np.inf)

        aux = dict(
            mu=mu, sig=sig, pi=pi, ei=ei, ei_norm=ei_norm, ucb_impr=ucb_impr,
            pca_bonus=pca_bonus, pc_redundancy_pen=pc_redundancy_pen,
            dmin=dmin, bound_close=bound_close,
        )
        return score, aux

    # probe sets
    Xp_sob = gen_sobol(probe_per_mode)
    Xp_pca = gen_pca(probe_per_mode)
    # local sigma is policy-adjusted: exploit => tighter, explore => wider
    sigma_loc = float(local_sigma * (1.15 if policy["action_name"] == "EXPLORE" else 0.85 if policy["action_name"] == "EXPLOIT" else 1.0))
    Xp_loc = gen_local(probe_per_mode, sigma=sigma_loc)

    sc_sob, _ = score_batch(Xp_sob)
    sc_pca, _ = score_batch(Xp_pca)
    sc_loc, _ = score_batch(Xp_loc)

    def reward_proxy(sc: np.ndarray) -> float:
        sc = sc[np.isfinite(sc)]
        if len(sc) == 0:
            return -1e9
        q = max(25, int(0.01 * len(sc)))  # top 1%
        top = np.partition(sc, -q)[-q:]
        return float(np.mean(top))

    mode_reward = {
        "sobol": reward_proxy(sc_sob),
        "pca":   reward_proxy(sc_pca),
        "local": reward_proxy(sc_loc),
    }

    # epsilon-greedy over mode preference (final-round safety)
    # with small prob, boost the weakest mode to keep exploration alive
    if rng.rand() < eps_mode:
        weakest = min(mode_reward.items(), key=lambda kv: kv[1])[0]
        mode_reward[weakest] += 0.20 * (max(mode_reward.values()) - min(mode_reward.values()) + 1e-6)

    # allocate full budget adaptively (MAB-style)
    # also bias local fraction from policy
    alloc = _mab_allocate_budgets_ucb(mode_reward, total=n_total, t=n, explore_frac=0.10)
    # enforce policy local fraction gently
    target_local = int(n_total * float(policy["params"]["local_frac"]))
    delta = target_local - alloc["local"]
    # shift delta from the best non-local mode
    if delta != 0:
        non_local = ["sobol", "pca"]
        best_non_local = max(non_local, key=lambda m: alloc[m])
        take = int(np.clip(delta, -alloc["local"], alloc[best_non_local]))
        alloc["local"] += take
        alloc[best_non_local] -= take

    n_sobol = max(5000, int(alloc["sobol"]))
    n_pca   = max(5000, int(alloc["pca"]))
    n_local = max(5000, int(alloc["local"]))

    # generate full candidate pool
    X_sob = gen_sobol(n_sobol)
    X_pca = gen_pca(n_pca)
    X_loc = gen_local(n_local, sigma=sigma_loc)
    Xcand = clip_to_bounds(np.vstack([X_sob, X_pca, X_loc]), lower, upper)

    # final scoring and pick
    score, aux = score_batch(Xcand)

    # relax once if too constrained
    if not np.isfinite(score).any():
        dmat = np.linalg.norm(Xcand[:, None, :] - Xc[None, :, :], axis=2)
        dmin = dmat.min(axis=1)
        ok2 = dmin >= (0.5 * min_dist)
        score = np.where(ok2, score, -np.inf)

    next_idx = int(np.argmax(score))
    next_x = Xcand[next_idx].copy()
    next_x_6 = round6(next_x)

    pick_mode = f"Week13(RL): Q={policy['action_name']} | alloc={alloc} | mode_reward={mode_reward}"

    # Rounded non-duplicate enforcement
    if enforce_rounded_nonduplicate:
        X_obs_6 = round6(clip_to_bounds(X_obs, lower, upper))
        seen = set(map(tuple, X_obs_6))
        if tuple(next_x_6) in seen:
            order = np.argsort(score)[::-1]
            for j in order[:30000]:
                cand6 = round6(Xcand[j])
                if tuple(cand6) not in seen:
                    next_idx = int(j)
                    next_x = Xcand[next_idx].copy()
                    next_x_6 = cand6
                    pick_mode += " + nondup-fallback"
                    break

    # nearest neighbors for audit
    dists = np.linalg.norm(Xc - next_x, axis=1)
    nn = np.argsort(dists)[:3]

    # interpretability: PC-weighted variable importance
    load_abs = np.abs(comps[:k, :])
    pc_w = evr[:k].reshape(-1, 1)
    var_importance = (pc_w * load_abs).sum(axis=0)
    var_importance = var_importance / max(float(var_importance.sum()), 1e-12)

    # decomp for transparency at selected point
    mu = aux["mu"][next_idx]
    sig = aux["sig"][next_idx]
    pi = aux["pi"][next_idx]
    ei = aux["ei"][next_idx]
    ei_norm = aux["ei_norm"][next_idx]
    ucb_impr = aux["ucb_impr"][next_idx]
    pca_bonus = aux["pca_bonus"][next_idx]
    pc_red = aux["pc_redundancy_pen"][next_idx]
    dmin = aux["dmin"][next_idx]
    bound_close = aux["bound_close"][next_idx]

    decomp = dict(
        score=float(score[next_idx]),
        xi=float(xi),
        beta=float(beta),
        alpha_ucb=float(alpha_ucb),
        ei=float(ei),
        ei_norm=float(ei_norm),
        ucb_impr=float(ucb_impr),
        pca_bonus=float(pca_bonus),
        pc_redundancy_pen=float(pc_red),
        dmin=float(dmin),
        bound_close=float(bound_close),
        pick_mode=pick_mode,
        policy_action=policy["action_name"],
        alloc=alloc,
        mode_reward=mode_reward,
        Q_table=policy["Q"],
    )

    reasoning = [
        "WEEK 13 RL NOTES (Function 3)",
        "",
        "Exploration–Exploitation (MAB + policy):",
        f"  - Offline Q-learning inferred regime => action={policy['action_name']} (state={policy['state_now']})",
        f"  - Acquisition adjusted: xi={xi:.6f}, beta={beta:.4f}, alpha_ucb={alpha_ucb:.3f}",
        "  - MAB allocation uses per-mode reward proxy (top-score mean) then allocates candidate budget.",
        f"  - Candidate allocation: sobol={n_sobol}, pca={n_pca}, local={n_local}",
        "",
        "Feedback-driven says:",
        "  - Best-so-far improvements in your history are treated as rewards;",
        "    stagnation increases exploration pressure, improvement increases exploitation pressure.",
        "",
        "AlphaGo Zero analogy:",
        "  - 'Self-play' here is surrogate-vs-surrogate: generate candidates, score them,",
        "    and reallocate search effort autonomously (no hand-tuned fixed split).",
        "  - This is mostly model-based planning (using the surrogate) with model-free flavor",
        "    in the Q update derived from observed improvements.",
        "",
        "PCA drivers (still used as a representation):",
        f"  - Explained variance ratio: {evr}",
        f"  - PC-weighted variable importance: {var_importance}",
        "",
        "Chosen point (audit):",
        f"  - x_next_6dp={next_x_6}",
        f"  - mu={float(mu):.6f}, sigma={float(sig):.6f}, PI={float(pi):.4f}, EI={float(ei):.6f}",
        f"  - score={float(score[next_idx]):.6f} (includes repulsion/boundary/PC redundancy)",
        "",
        "Nearest tested points:",
    ]
    for i, idx in enumerate(nn, 1):
        reasoning.append(
            f"  #{i}: x={round6(X_obs[idx])}, y={float(y_obs[idx]):.6f}, dist={float(dists[idx]):.6f}"
        )

    return dict(
        next_x=next_x_6,
        next_x_raw=next_x,
        pred_mean=float(mu),
        pred_std=float(sig),
        y_best=y_best,
        x_best=round6(X_obs[best_idx]),
        pca_evr=evr,
        pca_components=comps,
        var_importance=var_importance,
        decomp=decomp,
        reasoning="\n".join(reasoning),
    )

# ---------------------------------------------------------
# 7. Hyperparameter tuning (kept as-is from Week 11/12)
# ---------------------------------------------------------
def cv_mse_score(
    config: Dict,
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    k: int = 5,
    seed: int = 0
) -> float:
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    mses = []
    lower, upper = bounds

    for tr_idx, va_idx in kf.split(X):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        surr = MCDropoutSurrogate(
            hidden_layer_sizes=config["hidden"],
            dropout=config["dropout"],
            n_epochs=config["epochs"],
            lr=config["lr"],
            weight_decay=config["weight_decay"],
            random_state=seed,
            n_mc_samples=config["n_mc_samples"],
            robust_y=True
        )

        surr.fit(Xtr, ytr, lower, upper)
        preds = surr.predict(Xva, lower, upper)
        mses.append(np.mean((preds - yva) ** 2))

    return float(np.mean(mses))

def tune_hyperparameters(
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 8
) -> Dict:
    rng = np.random.RandomState(random_state)

    hidden_options = [(64, 32), (64, 64), (128, 64)]
    dropout_options = [0.10, 0.15, 0.20]
    lr_options = [7e-4, 1e-3, 2e-3]
    wd_options = [0.0, 1e-6, 1e-5]

    n_initial = 10
    configs = []
    for _ in range(n_initial):
        cfg = dict(
            hidden=hidden_options[rng.randint(len(hidden_options))],
            dropout=float(dropout_options[rng.randint(len(dropout_options))]),
            lr=float(lr_options[rng.randint(len(lr_options))]),
            weight_decay=float(wd_options[rng.randint(len(wd_options))]),
            n_mc_samples=int([96, 128][rng.randint(2)]),
        )
        configs.append(cfg)

    stage_epochs = [450, 1050]
    keep_fracs = [0.5, 0.4]
    best_overall = None

    for stage, (epochs, keep_frac) in enumerate(zip(stage_epochs, keep_fracs), start=1):
        scored = []
        for cfg in configs:
            cfg_stage = dict(cfg)
            cfg_stage["epochs"] = epochs
            mse = cv_mse_score(cfg_stage, X, y, bounds=bounds, k=5, seed=0)
            scored.append((mse, cfg_stage))

        scored.sort(key=lambda t: t[0])
        if best_overall is None or scored[0][0] < best_overall[0]:
            best_overall = scored[0]

        k_keep = max(4, int(len(scored) * keep_frac))
        configs = [cfg for _, cfg in scored[:k_keep]]

        print(f"\n--- TUNING STAGE {stage} ---")
        print(f"epochs={epochs}, kept={k_keep}/{len(scored)}")
        print(f"best CV-MSE so far: {best_overall[0]:.6f}")
        print(f"best config so far: {best_overall[1]}")

    return dict(best_cv_mse=best_overall[0], best_config=best_overall[1])

# ---------------------------------------------------------
# 8. Main
# ---------------------------------------------------------
def main():
    np.random.seed(0)
    torch.manual_seed(0)

    bounds = (np.zeros(3), np.ones(3))
    lower, upper = bounds

    print("================================================")
    print("WEEK 13 FUNCTION 3 — RL PRE-CHECKS")
    print("================================================")
    print(f"Points (n): {len(X_raw)}")
    print(f"Frac out-of-bounds in X_raw (before clip): {frac_out_of_bounds(X_raw, lower, upper):.3f}")

    # Tune hyperparameters
    tuning = tune_hyperparameters(X_raw, y_raw, bounds=bounds, random_state=8)
    best_cfg = tuning["best_config"]

    # Fit ensemble
    seeds = [0, 11, 29]  # fixed for determinism
    members = []
    for s in seeds:
        members.append(
            MCDropoutSurrogate(
                hidden_layer_sizes=best_cfg["hidden"],
                dropout=best_cfg["dropout"],
                n_epochs=best_cfg["epochs"],
                lr=best_cfg["lr"],
                weight_decay=best_cfg["weight_decay"],
                random_state=s,
                n_mc_samples=best_cfg["n_mc_samples"],
                robust_y=True
            )
        )

    surrogate = EnsembleSurrogate(members=members)
    surrogate.fit(X_raw, y_raw, lower, upper)

    # Current best
    best_idx = int(np.argmax(y_raw))
    current_best_x = round6(clip_to_bounds(X_raw[best_idx], lower, upper))
    current_best_y = float(y_raw[best_idx])

    # Propose next query (Week 13 RL)
    suggestion = propose_next_point_week13(
        surrogate=surrogate,
        X_obs=X_raw,
        y_obs=y_raw,
        bounds=bounds,
        random_state=123,

        n_total=110_000,

        min_dist=0.010,
        repulse_len=0.050,
        repulse_w=0.28,
        bound_margin=0.006,

        xi_base=0.0008,
        alpha_ucb_base=0.30,

        pca_dims=2,
        pca_bonus_w=0.020,
        pc_redundancy_w=0.020,

        local_sigma=0.040,
        probe_per_mode=6000,
        eps_mode=0.05,

        enforce_rounded_nonduplicate=True
    )

    x_next = suggestion["next_x"]

    print("\n================================================")
    print("WEEK 13 FUNCTION 3 — NEXT POINT (<= 6 DECIMALS)")
    print("================================================")
    print("Surrogate: ensemble(mc_dropout) + robust y scaling")
    print(f"Best CV-MSE (lower is better): {tuning['best_cv_mse']:.6f}")
    print("Best tuned hyperparameters:")
    print(f"  hidden: {best_cfg['hidden']}")
    print(f"  dropout: {best_cfg['dropout']}")
    print(f"  lr: {best_cfg['lr']}")
    print(f"  weight_decay: {best_cfg['weight_decay']}")
    print(f"  n_mc_samples: {best_cfg['n_mc_samples']}")
    print(f"  epochs: {best_cfg['epochs']}")
    print(f"Ensemble seeds: {seeds}")

    print("\n================================================")
    print("CURRENT BEST OBSERVED")
    print("================================================")
    print(f"x_best = {as_fixed6_list(current_best_x)}, y_best = {current_best_y:.6f}")

    print("\n================================================")
    print("RECOMMENDED NEXT POINT (rounded to 6 decimals)")
    print("================================================")
    print(f"x_next     = [{x_next[0]:.6f}, {x_next[1]:.6f}, {x_next[2]:.6f}]")
    print(f"mu(x_next) = {suggestion['pred_mean']:.6f}")
    print(f"sd(x_next) = {suggestion['pred_std']:.6f}")

    print("\n================================================")
    print("RL + MAB TRANSPARENCY")
    print("================================================")
    d = suggestion["decomp"]
    print(f"policy_action  = {d['policy_action']}")
    print(f"xi             = {d['xi']:.6f}")
    print(f"beta           = {d['beta']:.4f}")
    print(f"alpha_ucb      = {d['alpha_ucb']:.3f}")
    print(f"alloc          = {d['alloc']}")
    print(f"mode_reward    = {d['mode_reward']}")
    print(f"Q_table(2x3)   =\n{d['Q_table']}")

    print("\n================================================")
    print("PCA SUMMARY (TRANSPARENCY)")
    print("================================================")
    evr = suggestion["pca_evr"]
    vi = suggestion["var_importance"]
    print(f"Explained variance ratio (PC1..PC3): {evr}")
    print(f"PC-weighted variable importance (dims 1..3): {vi}  (normalized)")

    print("\n================================================")
    print("SCORE DECOMPOSITION @ x_next")
    print("================================================")
    print(f"score          = {d['score']:.6f}")
    print(f"EI             = {d['ei']:.6f} (EI_norm={d['ei_norm']:.6f})")
    print(f"UCB_impr       = {d['ucb_impr']:.6f}")
    print(f"pca_bonus      = {d['pca_bonus']:.6f}")
    print(f"pc_redundancy  = {d['pc_redundancy_pen']:.6f}")
    print(f"dmin           = {d['dmin']:.6f}")
    print(f"bound_close    = {d['bound_close']:.6f}")

    print("\n================================================")
    print("REASONING (AUDIT TRAIL)")
    print("================================================")
    print(suggestion["reasoning"])

if __name__ == "__main__":
    main()

WEEK 13 FUNCTION 3 — RL PRE-CHECKS
Points (n): 27
Frac out-of-bounds in X_raw (before clip): 0.037

--- TUNING STAGE 1 ---
epochs=450, kept=5/10
best CV-MSE so far: 0.022606
best config so far: {'hidden': (64, 32), 'dropout': 0.2, 'lr': 0.0007, 'weight_decay': 1e-06, 'n_mc_samples': 128, 'epochs': 450}

--- TUNING STAGE 2 ---
epochs=1050, kept=4/5
best CV-MSE so far: 0.022386
best config so far: {'hidden': (64, 32), 'dropout': 0.2, 'lr': 0.0007, 'weight_decay': 1e-06, 'n_mc_samples': 128, 'epochs': 1050}

WEEK 13 FUNCTION 3 — NEXT POINT (<= 6 DECIMALS)
Surrogate: ensemble(mc_dropout) + robust y scaling
Best CV-MSE (lower is better): 0.022386
Best tuned hyperparameters:
  hidden: (64, 32)
  dropout: 0.2
  lr: 0.0007
  weight_decay: 1e-06
  n_mc_samples: 128
  epochs: 1050
Ensemble seeds: [0, 11, 29]

CURRENT BEST OBSERVED
x_best = [0.403756, 0.381706, 0.489738], y_best = -0.009136

RECOMMENDED NEXT POINT (rounded to 6 decimals)
x_next     = [1.000000, 1.000000, 0.470937]
mu(x_next) = -0